
# mt5-small (zh→en) — Baseline TF Training (Wuxia Domain)

**TFG Anonymous – Baseline NMT (MarianMT)**  
This notebook trains and evaluates to model **MarianMT** (`google/mt5-small`)  using to dataset of **wuxia** (Chinese→English) already preparado in format `datasets` (HF).




## 1) Environment of execution and installation of dependencies

In [ ]:

import os, random, math
import numpy as np

import torch
print("CUDA disponible:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Name of the GPU:", torch.cuda.get_device_name(0))




> **Requirements of the dataset**: directory HF Datasets with *splits* `train`, `validation`, `test` and columns `zh` (Chinese) and `en` (English):  
> `processed_data/wuxia_zh_en_clean/`

In [ ]:
# Configuration of carpetas for entorno LOCAL
from pathlib import Path
BASE_DIR = Path.cwd().parent
BASE_DIR.mkdir(exist_ok=True)

# Structure requerida by the user
for sub in ["training", "models", "processed_data"]:
    (BASE_DIR / sub).mkdir(parents=True, exist_ok=True)

print("Base:", BASE_DIR.resolve())
print("Structure created (if not existed):")
for p in ["trainign", "models", "proccesed data"]:
    print(" -", (BASE_DIR / p).resolve())

# Score: the dataset must existir in: CORPUS/proccesed data/wuxia_zh_en_clean


## 2) Configuration

In [ ]:
from dataclasses import dataclass

@dataclass
class Config:
    # Paths (local)
    dataset_dir: Path  = BASE_DIR / "processed_data" / "wuxia_zh_en_clean"   
    output_dir: Path   = BASE_DIR / "models" / "mt5_small"             
    ckpt_dir: Path     = BASE_DIR / "checkpoints"
    training_dir: Path = BASE_DIR / "training"

    # Columns of the dataset
    src_col: str = "zh"
    tgt_col: str = "en"

    # Model 
    model_ckpt: str = "google/mt5-small"

    # Prefix of instruction for T5/mT5 
    use_instruction_prefix: bool = True
    translation_prefix: str = "translate Chinese to English: "

    # Training
    seed: int = 42
    max_source_length: int = 128
    max_target_length: int = 128
    batch_size: int = 16
    epochs: int = 10
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    early_stopping_patience: int = 3
    fraction: float = 1

cfg = Config()
print(cfg)


In [ ]:
import random, numpy as np, os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)



# Semillas for 
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
random.seed(cfg.seed)
os.environ["PYTHONHASHSEED"] = str(cfg.seed)


if device.type == "cuda":
    torch.cuda.manual_seed_all(cfg.seed)
    # For reproducibilidad estricta 
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Semillas fijadas and backend configured.")


## 3) Load dataset (Hugging Face Datasets)

In [ ]:

from datasets import load_from_disk, DatasetDict

assert os.path.isdir(cfg.dataset_dir), f"The dataset was not found at: {cfg.dataset_dir}"
raw_ds: DatasetDict = load_from_disk(cfg.dataset_dir)
print(raw_ds)

# Validar columns
def _check_cols(ds, src_col, tgt_col, split):
    cols = ds.column_names
    assert src_col in cols and tgt_col in cols, f"El split '{split}' must contener columns '{src_col}' y '{tgt_col}'. Columns: {cols}"

for split in ["train", "validation", "test"]:
    assert split in raw_ds, f"Falta el split '{split}' en el dataset."
    _check_cols(raw_ds[split], cfg.src_col, cfg.tgt_col, split)

# Submuestreo 
def take_fraction(ds, frac, seed=42):
    if frac >= 1.0:
        return ds
    n = max(1, int(len(ds) * frac))
    return ds.shuffle(seed=seed).select(range(n))

train_ds = take_fraction(raw_ds["train"], cfg.fraction, seed=cfg.seed)
val_ds   = take_fraction(raw_ds["validation"], cfg.fraction, seed=cfg.seed)
test_ds  = take_fraction(raw_ds["test"], cfg.fraction, seed=cfg.seed)

print(train_ds[:2])
print(f"Tam. train/val/test (fraction={cfg.fraction}):", len(train_ds), len(val_ds), len(test_ds))


## 4) Tokenizer

In [ ]:

from transformers import MT5Tokenizer, MT5ForConditionalGeneration

# Tokenizer
tokenizer = MT5Tokenizer.from_pretrained(cfg.model_ckpt)

# Model
model = MT5ForConditionalGeneration.from_pretrained(cfg.model_ckpt)

#  tokens special <id_0> etc
# Pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Decoder start token
if model.config.decoder_start_token_id is None:
    model.config.decoder_start_token_id = tokenizer.pad_token_id

# EOS token
if model.config.eos_token_id is None:
    model.config.eos_token_id = tokenizer.eos_token_id

model.config.use_cache = False

# Enviar a dispositivo
model.to(device)

# Info
print("Model:", cfg.model_ckpt)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters total: {n_params:,}")
print("pad_token_id:", tokenizer.pad_token_id)
print("eos_token_id:", tokenizer.eos_token_id)
print("decoder_start_token_id:", model.config.decoder_start_token_id)


## 5) Preprocesamiento

In [ ]:

max_source_length = cfg.max_source_length
max_target_length = cfg.max_target_length

def preprocess_function(examples):
    # Source (ZH) with prefix of instruction for mT5
    if cfg.use_instruction_prefix:
        src_texts = [cfg.translation_prefix + s.strip() for s in examples[cfg.src_col]]
    else:
        src_texts = [s.strip() for s in examples[cfg.src_col]]

    # Target (IN)
    tgt_texts = [t.strip() for t in examples[cfg.tgt_col]]

    # 3) Tokenization: text_target=
    model_inputs = tokenizer(
        src_texts,
        max_length=max_source_length,
        truncation=True,
        padding=False
    )
    labels = tokenizer(
        text_target=tgt_texts,           # important for that assumes labels bien
        max_length=max_target_length,
        truncation=True,
        padding=False
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = raw_ds.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_ds["train"].column_names
)

train_ds = take_fraction(tokenized_datasets["train"], cfg.fraction, seed=cfg.seed)
val_ds   = take_fraction(tokenized_datasets["validation"], cfg.fraction, seed=cfg.seed)
test_ds  = take_fraction(tokenized_datasets["test"], cfg.fraction, seed=cfg.seed) if "test" in tokenized_datasets else val_ds

print("Example tokenized:", {k: type(v) for k,v in train_ds[0].items()})


## 6) Data collator

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding="longest",
    pad_to_multiple_of=8,   # GPU
    return_tensors="pt"
)

# Example of batch
batch = data_collator([train_ds[i] for i in range(2)])
for k, v in batch.items():
    print(f"{k}: shape={v.shape}, dtype={v.dtype}")


## 7) Configuration of the training


> **By default**          
> **Optimizador**: `AdamW` (with LR=2e-5, weight decay=0.01) of `Seq2SeqTrainer`   
> **Loss**: `CrossEntropyLoss` (token-level) of `AutoModelForSeq2SeqLM `





In [ ]:
from transformers import Seq2SeqTrainer

class MySeq2SeqTrainer(Seq2SeqTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """
        Sobrescribe compute_loss for ensure that siempre is usan the labels. Evita <id_0>
        Ignora arguments extra (e.g.. num_items_in_batch) that the model not accepts.
        """
        labels = inputs.get("labels")

        # forward only with the arguments that mT5 accepts
        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs.get("attention_mask"),
            labels=labels,
        )

        loss = outputs.loss
        return (loss, outputs) if return_outputs else loss


In [ ]:

from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

# Directory for results
run_dir = cfg.output_dir
run_dir.mkdir(parents=True, exist_ok=True)

# Arguments of training 
training_args = Seq2SeqTrainingArguments(
    output_dir=str(run_dir),
    overwrite_output_dir=True,
    eval_strategy="epoch",          
    save_strategy="epoch",
    save_total_limit=3,
    learning_rate=cfg.learning_rate,
    per_device_train_batch_size=cfg.batch_size,
    per_device_eval_batch_size=cfg.batch_size,
    num_train_epochs=cfg.epochs,
    weight_decay=cfg.weight_decay,
    logging_steps=200,
    predict_with_generate=True,           # <- important for  translation
    generation_max_length=cfg.max_target_length,
    generation_num_beams=4,
    fp16=False, #torch.cuda.is_available(), In this case of mt5 NOT is uses fp16 by reasons of speed/optimizacion
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    max_grad_norm=1.0,
)

# Trainer for Seq2Seq
trainer = MySeq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator
)


print(" Seq2SeqTrainer configured (PyTorch) with mT5.")


## 8) Training

In [ ]:

from transformers import EarlyStoppingCallback
import json

# Add callback of early stopping
trainer.add_callback(EarlyStoppingCallback(
    early_stopping_patience=cfg.early_stopping_patience,  # number of evaluations without improvement
    early_stopping_threshold=0.0
))

# Entrenar
train_result = trainer.train()

# Save model final and tokenizer
trainer.save_model(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)

# Save metrics of training
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
trainer.save_state()

# Save configuration and results in JSON 
run_info = {
    "model_name": cfg.model_ckpt,
    "epochs": cfg.epochs,
    "batch_size": cfg.batch_size,
    "learning_rate": cfg.learning_rate,
    "weight_decay": cfg.weight_decay,
    "train_size": len(train_ds),
    "val_size": len(val_ds),
    "metrics": metrics
}

with open(cfg.output_dir / "run_info.json", "w", encoding="utf-8") as f:
    json.dump(run_info, f, indent=4, ensure_ascii=False)

print(" Training finished, model and artefactos saved in:", cfg.output_dir)
